In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("USE DATABASE HEALTHPULSE_DB").collect()
session.sql("USE SCHEMA GOLD_STAGE").collect()

session.sql("SELECT CURRENT_DATABASE(), CURRENT_SCHEMA(), CURRENT_WAREHOUSE()").collect()

In [ ]:
f  = session.table("FACT_APPOINTMENTS")
p  = session.table("DIM_PATIENTS")
pr = session.table("DIM_PROVIDERS")
c  = session.table("DIM_CLINICS")
d  = session.table("DIM_DATES")

In [ ]:
print("FACT:", f.columns)
print("PATIENTS:", p.columns)
print("PROVIDERS:", pr.columns)
print("CLINICS:", c.columns)
print("DATES:", d.columns)

In [ ]:
joined = (
    f
    .join(p,  f["PATIENT_ID"] == p["PATIENT_ID"], "left")
    .join(pr, f["PROVIDER_ID"] == pr["PROVIDER_ID"], "left")
    .join(c,  f["PROVIDER_CLINIC_ID"] == c["PROVIDER_CLINIC_ID"], "left")
    .join(d,  f["DATE_ID"] == d["DATE_ID"], "left")
)

joined.limit(5).collect()

In [ ]:
model_df = joined.select(
    f["APPOINTMENT_ID"].as_("APPOINTMENT_ID"),
    f["PATIENT_ID"].as_("PATIENT_ID"),
    f["PROVIDER_ID"].as_("PROVIDER_ID"),
    f["PROVIDER_CLINIC_ID"].as_("PROVIDER_CLINIC_ID"),
    f["DATE_ID"].as_("DATE_ID"),

    f["APPOINTMENT_TIME"].as_("APPOINTMENT_TIME"),
    f["LEAD_TIME_DAYS"].as_("LEAD_TIME_DAYS"),
    f["WAIT_TIME_MINUTES"].as_("WAIT_TIME_MINUTES"),
    f["IS_NO_SHOW"].as_("NO_SHOW"),   # ✅ target

    c["CLINIC_NAME"].as_("CLINIC_NAME"),
    c["CITY"].as_("CITY"),
    c["HOURS"].as_("HOURS")
)

model_df.limit(10).collect()

In [ ]:
# 1) Quick preview to confirm the dataset is correct
model_df.limit(10).collect()

In [ ]:
# 2) Row count (takes a moment but good to know)
model_df.count()

In [ ]:
from snowflake.snowpark.functions import col

model_df.group_by(col("NO_SHOW")).count().sort(col("NO_SHOW")).collect()

In [ ]:
pdf = model_df.to_pandas()
pdf.shape, pdf.head()

In [ ]:
# target distribution
print(pdf["NO_SHOW"].value_counts(dropna=False))
print("\nNo-show rate:", pdf["NO_SHOW"].mean())

# missing values
pdf.isna().sum().sort_values(ascending=False).head(12)

In [ ]:
pdf["WAIT_TIME_MINUTES"] = pdf["WAIT_TIME_MINUTES"].fillna(
    pdf["WAIT_TIME_MINUTES"].median()
)

In [ ]:
pdf["WAIT_TIME_MINUTES"].isna().sum()

In [1]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

num_features = ["LEAD_TIME_DAYS", "WAIT_TIME_MINUTES"]
cat_features = ["CLINIC_NAME", "CITY", "HOURS", "APPOINTMENT_TIME"]

X = pdf[num_features + cat_features]
y = pdf["NO_SHOW"]

prep = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
    ]
)

model = Pipeline([
    ("prep", prep),
    ("clf", LogisticRegression(max_iter=1000))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model.fit(X_train, y_train)
pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print("Confusion matrix:\n", confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))

NameError: name 'pdf' is not defined

In [ ]:
pdf.groupby("NO_SHOW")[["LEAD_TIME_DAYS", "WAIT_TIME_MINUTES"]].mean()

In [ ]:
pdf.groupby("CLINIC_NAME")["NO_SHOW"].mean().sort_values(ascending=False)

In [ ]:
pdf.groupby("CITY")["NO_SHOW"].mean().sort_values(ascending=False)

In [ ]:
pdf.groupby("APPOINTMENT_TIME")["NO_SHOW"].mean().sort_values(ascending=False)

In [ ]:
pdf["LEAD_TIME_DAYS"].describe()
pdf["WAIT_TIME_MINUTES"].describe()

In [ ]:
LogisticRegression(class_weight="balanced")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
pdf.head()

In [ ]:
pdf.shape

In [ ]:
CREATE OR REPLACE TEMP TABLE MODEL_DATA AS
SELECT 
    f.APPOINTMENT_ID,
    f.PATIENT_ID,
    f.PROVIDER_ID,
    f.PROVIDER_CLINIC_ID,
    f.DATE_ID,
    f.APPOINTMENT_TIME,
    f.LEAD_TIME_DAYS,
    f.WAIT_TIME_MINUTES,
    f.IS_NO_SHOW AS NO_SHOW,
    c.CLINIC_NAME,
    c.CITY,
    c.HOURS
FROM HEALTHPULSE_DB.GOLD_STAGE.FACT_APPOINTMENTS f
LEFT JOIN HEALTHPULSE_DB.GOLD_STAGE.DIM_CLINICS c
ON f.PROVIDER_CLINIC_ID = c.PROVIDER_CLINIC_ID;

In [ ]:
SELECT * 
FROM (
    SELECT 
        f.APPOINTMENT_ID,
        f.PATIENT_ID,
        f.PROVIDER_ID,
        f.PROVIDER_CLINIC_ID,
        f.DATE_ID,
        f.APPOINTMENT_TIME,
        f.LEAD_TIME_DAYS,
        f.WAIT_TIME_MINUTES,
        f.IS_NO_SHOW AS NO_SHOW,
        c.CLINIC_NAME,
        c.CITY,
        c.HOURS
    FROM HEALTHPULSE_DB.GOLD_STAGE.FACT_APPOINTMENTS f
    LEFT JOIN HEALTHPULSE_DB.GOLD_STAGE.DIM_CLINICS c
    ON f.PROVIDER_CLINIC_ID = c.PROVIDER_CLINIC_ID
)
LIMIT 120000;

In [ ]:
LogisticRegression(class_weight="balanced")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Features
num_features = ["LEAD_TIME_DAYS", "WAIT_TIME_MINUTES"]
cat_features = ["CLINIC_NAME", "CITY", "HOURS", "APPOINTMENT_TIME"]

X = pdf[num_features + cat_features]
y = pdf["NO_SHOW"]

# Pipeline
prep = ColumnTransformer([
    ("num", "passthrough", num_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features)
])

model = Pipeline([
    ("prep", prep),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train
model.fit(X_train, y_train)

# Predict
pred = model.predict(X_test)

# Results
print("Accuracy:", accuracy_score(y_test, pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))

In [ ]:
print(p.columns)
print(d.columns)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
print(p.columns)
print(d.columns)

In [ ]:
from snowflake.snowpark.functions import col

model_df = joined.select(
    f["APPOINTMENT_ID"],
    f["PATIENT_ID"],
    f["PROVIDER_ID"],
    f["PROVIDER_CLINIC_ID"],
    f["DATE_ID"],
    f["APPOINTMENT_TIME"],
    f["LEAD_TIME_DAYS"],
    f["WAIT_TIME_MINUTES"],
    f["IS_NO_SHOW"].as_("NO_SHOW"),

    # clinic
    c["CLINIC_NAME"],
    c["CITY"],
    c["HOURS"],

    # NEW FEATURES 🔥
    p["AGE"],
    p["INSURANCE_TYPE"],
    d["DAY_OF_WEEK"]
)

In [ ]:
f  = session.table("FACT_APPOINTMENTS")
p  = session.table("DIM_PATIENTS")
pr = session.table("DIM_PROVIDERS")
c  = session.table("DIM_CLINICS")
d  = session.table("DIM_DATES")

In [ ]:
joined = (
    f
    .join(p,  f["PATIENT_ID"] == p["PATIENT_ID"], "left")
    .join(pr, f["PROVIDER_ID"] == pr["PROVIDER_ID"], "left")
    .join(c,  f["PROVIDER_CLINIC_ID"] == c["PROVIDER_CLINIC_ID"], "left")
    .join(d,  f["DATE_ID"] == d["DATE_ID"], "left")
)

In [ ]:
model_df = joined.select(
    # keys + base features from FACT
    f["APPOINTMENT_ID"].as_("APPOINTMENT_ID"),
    f["PATIENT_ID"].as_("PATIENT_ID"),
    f["PROVIDER_ID"].as_("PROVIDER_ID"),
    f["PROVIDER_CLINIC_ID"].as_("PROVIDER_CLINIC_ID"),
    f["DATE_ID"].as_("DATE_ID"),
    f["APPOINTMENT_TIME"].as_("APPOINTMENT_TIME"),
    f["LEAD_TIME_DAYS"].as_("LEAD_TIME_DAYS"),
    f["WAIT_TIME_MINUTES"].as_("WAIT_TIME_MINUTES"),
    f["IS_NO_SHOW"].as_("NO_SHOW"),

    # clinic
    c["CLINIC_NAME"].as_("CLINIC_NAME"),
    c["CITY"].as_("CITY"),
    c["HOURS"].as_("HOURS"),

    # patient features
    p["AGE"].as_("AGE"),
    p["INSURANCE_TYPE"].as_("INSURANCE_TYPE"),

    # date features
    d["DAY_OF_WEEK"].as_("DAY_OF_WEEK"),
    d["APPOINTMENT_DATE"].as_("APPOINTMENT_DATE")
)

In [ ]:
pdf = model_df.to_pandas()
pdf.shape

In [ ]:
pdf["WAIT_TIME_MINUTES"] = pdf["WAIT_TIME_MINUTES"].fillna(pdf["WAIT_TIME_MINUTES"].median())

In [ ]:
print(model_df.columns)

In [ ]:
import pandas as pd

# Fill missing
pdf["WAIT_TIME_MINUTES"] = pdf["WAIT_TIME_MINUTES"].fillna(pdf["WAIT_TIME_MINUTES"].median())

# Convert date
pdf["APPOINTMENT_DATE"] = pd.to_datetime(pdf["APPOINTMENT_DATE"], errors="coerce")

# Create features
pdf["MONTH"] = pdf["APPOINTMENT_DATE"].dt.month
pdf["DAY"] = pdf["APPOINTMENT_DATE"].dt.day

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

num_features = ["LEAD_TIME_DAYS", "WAIT_TIME_MINUTES", "AGE", "MONTH", "DAY"]
cat_features = ["CLINIC_NAME", "CITY", "HOURS", "APPOINTMENT_TIME", "INSURANCE_TYPE", "DAY_OF_WEEK"]

X = pdf[num_features + cat_features]
y = pdf["NO_SHOW"]

prep = ColumnTransformer(
    transformers=[
        ("num", "passthrough", num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_features),
    ]
)

model = Pipeline([
    ("prep", prep),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

model.fit(X_train, y_train)
pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, pred))
print(classification_report(y_test, pred))

In [ ]:
import numpy as np
from sklearn.metrics import precision_recall_fscore_support

proba = model.predict_proba(X_test)[:, 1]

for t in [0.2, 0.3, 0.4, 0.5, 0.6]:
    pred_t = (proba >= t).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_test, pred_t, average="binary")
    print(f"threshold={t:.1f}  precision={p:.2f}  recall={r:.2f}  f1={f1:.2f}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = Pipeline([
    ("prep", prep),
    ("clf", RandomForestClassifier(
        n_estimators=300,
        random_state=42,
        class_weight="balanced_subsample",
        n_jobs=-1
    ))
])

rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)

print("RF Accuracy:", accuracy_score(y_test, pred_rf))
print("RF Confusion Matrix:\n", confusion_matrix(y_test, pred_rf))
print(classification_report(y_test, pred_rf))

In [ ]:
print(X.columns)

In [ ]:
import numpy as np

rf_model = rf.named_steps["clf"]
ohe = rf.named_steps["prep"].named_transformers_["cat"]

cat_names = ohe.get_feature_names_out(cat_features)
feature_names = np.array(num_features + list(cat_names))

importances = rf_model.feature_importances_

top_idx = np.argsort(importances)[::-1][:15]

list(zip(feature_names[top_idx], importances[top_idx]))